# Pre-process (resize)

In [ ]:
SKIP_PREPROCESSING = False

In [ ]:
import subprocess
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from glob import glob


def resize_video_gpu(input_path: str, output_path: str, short_side: int = 224) -> bool:
    """Resize a video so its shorter side = short_side, preserving aspect ratio."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    # Scale filter: shorter side = 224, -2 = auto (divisible by 2)
    scale_filter = f"scale_cuda='if(gt(iw,ih),-2,{short_side})':'if(gt(iw,ih),{short_side},-2)'"
    cmd = [
        "ffmpeg", "-y",
        "-hwaccel", "cuda",               # GPU decode
        "-hwaccel_output_format", "cuda", # keep frames on GPU VRAM
        "-i", input_path,
        "-vf", scale_filter,              # scale on GPU
        "-c:v", "h264_nvenc",             # GPU encode → MP4
        "-cq", "18",                      # high quality
        "-an",                            # drop audio
        output_path
    ]
    result = subprocess.run(cmd, capture_output=True)
    return result.returncode == 0


def resize_dataset(input_pattern: str, output_root: str, short_side: int, num_threads: int):
    tasks = []
    total_size_gb = 0
    for vid in glob(input_pattern, recursive=True):
        out = Path(output_root) / os.path.basename(vid)
        tasks.append((vid, str(out)))
        total_size_gb += round(os.path.getsize(vid) / (1024**3), 2)

    print(f"Found {len(tasks)} videos to resize.")
    print(f"Total size: {total_size_gb:.2f} GB")
    failed = []
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = {executor.submit(resize_video_gpu, inp, out, short_side): inp
                for inp, out in tasks}
        with tqdm(total=round(total_size_gb, 2), unit="GB", desc="Resizing videos") as pbar:
            for future in as_completed(futures):
                inp = futures[future]
                if not future.result():
                    failed.append(inp)
                pbar.update(round(os.path.getsize(inp) / (1024**3), 2))

    print(f"Done. Failed: {len(failed)}")
    if failed:
        print("\n".join(failed))

In [ ]:
if not SKIP_PREPROCESSING:
    resize_dataset(
        input_pattern="/kaggle/input/datasets/ldthanh/epic-kitchens-100-p*/*/*/*.MP4",
        output_root="/tmp/epic-kitchens-100-224",
        short_side=224,
        num_threads=48,  # NVENC sessions (GPU handles the heavy work)
    )
else:
    print("[INFO] Skipping preprocessing step.")

# Setup

In [ ]:
!cp -r /kaggle/input/notebooks/ldthanh/prepare-vitra-resources/VITRA .

In [ ]:
!uv pip install scipy \
  "torch" \
  "torchvision" \
  "torchaudio" \
  "PyYAML==6.0.2" \
  "hydra-colorlog>=1.2.0" \
  "hydra-core>=1.1.1" \
  "deepspeed==0.16.5" \
  "tensorboard>=2.13.0" \
  "tensorboardX>=2.6.2" \
  "tqdm>=4.65.0" \
  "transformers==4.47.1" \
  "diffusers>=0.31.0" \
  "wandb>=0.19.0" \
  "numpy<2.0" \
  "sentence-transformers==2.2.2" \
  "open_clip_torch==2.20.0" \
  "datasets==2.12.0" \
  "draccus==0.8.0" \
  "einops" \
  "huggingface_hub" \
  "json-numpy" \
  "jsonlines" \
  "matplotlib" \
  "peft==0.11.1" \
  "protobuf" \
  "rich" \
  "sentencepiece==0.1.99" \
  "timm==0.9.10" \
  "tokenizers>=0.21" \
  "decord" \
  "scikit-image" \
  "brotli" \
  "imageio-ffmpeg" \
  "imageio" \
  "ffmpeg-python" \
  "opencv-python" \
  "pympler" \
  "ultralytics" \
  "pytorch-lightning" \
  "yacs" \
  "utils3d" \
  --no-index --find-links "/kaggle/input/notebooks/ldthanh/prepare-vitra-resources/wheels"

In [ ]:
import os
os.chdir('VITRA')

In [ ]:
!mkdir -p data/VITRA_1M/{Video,Annotation}

if SKIP_PREPROCESSING:
    print("[INFO] Linking original videos since preprocessing was skipped.")
    !mkdir data/VITRA_1M/Video/Epic-Kitchen_root
    !for vid in /kaggle/input/datasets/ldthanh/epic-kitchens-100-p*/*/*/*.MP4; do \
        ln -sf $vid data/VITRA_1M/Video/Epic-Kitchen_root/`basename $vid`; done
else:
    print("[INFO] Linking resized videos since preprocessing was done.")
    !ln -s /tmp/epic-kitchens-100-224 data/VITRA_1M/Video/Epic-Kitchen_root

!ln -s /kaggle/input/datasets/nahidsiddique/something-something-v2/20bn-something-something-v2/20bn-something-something-v2/ data/VITRA_1M/Video/Somethingsomething-v2_root
!ln -s /kaggle/input/datasets/ldthanh/vitra-1m/epic/epic/ data/VITRA_1M/Annotation/epic
!ln -s /kaggle/input/datasets/ldthanh/vitra-1m/ssv2/ssv2/ data/VITRA_1M/Annotation/ssv2
!ln -s /kaggle/input/datasets/ldthanh/vitra-1m/statistics/ data/VITRA_1M/Annotation/statistics

# Patch code

In [ ]:
# vitra/configs/human_pretrain.json
content = """{
    "vla_name": "VITRA_Paligemma",
    "task_name": "pretrain",
    "model": "vitra_paligemma2",
    "fwd_pred_next_n": 16,
    "seed": 42,
    "batch_size": 64,
    "output_root": "data/vla_checkpoint/vitra_vla_3b/checkpoints",
    "log_root": "data/vla_checkpoint/vitra_vla_3b/logs",
    "cache_root": "data/vla_checkpoint/vitra_vla_3b/cache/",
    "model_load_path": "/kaggle/input/models/ldthanh/vitra-vla-3b/transformers/magic_mix_epic_ssv2/9/final-epoch=0-step=18000.ckpt",
    "resume": true,
    "wandb_project": "vitra_paligemma2_humanpretrain",
    "wandb_entity": "",
    "save_steps": 999999,
    "epoch_save_interval": 999999,
    "total_batch_size": 512,
    "use_bf16": true,
    "use_fov": true,
    "untied_cognition_token": true,
    "use_state": "DiT",
    "loss_type": "human",
    "train_setup": {
        "freeze_option": "freeze_vision_encoder"
    },
    "state_encoder": {
        "state_dim": 212
    },
    "action_model": {
        "model_type": "DiT-B",
        "token_size": 2304,
        "action_dim": 192,
        "hidden_size": 1024
    },
    "vlm": {
        "type": "PaliGemmaForConditionalGeneration",
        "name": "paligemma",
        "pretrained_model_name_or_path": "/kaggle/input/models/ldthanh/paligemma2-3b-mix-224/transformers/default/1"
    },
    "trainer": {
        "sharding_strategy": "shard-grad-op",
        "strategy": "fsdp_paligemma_with_checkpointing",
        "lr_scheduler_type": "backbone-freeze-warmup",
        "gradient_clip_val": 1.0,
        "learning_rate": 1e-05,
        "weight_decay": 0.1,
        "max_epochs": 100000,
        "max_steps": 20000,
        "reduce_in_full_precision": true,
        "enable_mixed_precision_training": false,
        "enable_gradient_checkpointing": true,
        "action_model_learning_rate": 1e-4,
        "llm_freeze_step": 625,
        "warmup_ratio": null
    },
    "train_dataset": {
        "data_root_dir": "data/VITRA_1M",
        "augmentation": true,
        "set_none_ratio": 0.0,
        "data_mix": "magic_mix_epic_ssv2",
        "num_workers": 32,
        "prefetch_factor": null,
        "flip_augmentation": 1.0,
        "action_type": "angle",
        "use_rel": false,
        "clip_len": null,
        "normalization": true,
        "state_mask_prob": 0.1
    },
    "repeated_diffusion_steps": 8
}
"""

with open('vitra/configs/human_pretrain.json', 'w') as f:
    f.write(content)

# Train

In [ ]:
# !rm -rf /kaggle/working/VITRA/data/vla_checkpoint
# !python scripts/train.py --config vitra/configs/human_pretrain.json

# Evaluate 

## Reference evaluation (old protocol)

In [ ]:
# !mkdir eval_results

# !python scripts/evaluate_pretrained_loss.py \
#     --config vitra/configs/human_pretrain.json \
#     --weights "/kaggle/input/models/ldthanh/vitra-vla-3b/transformers/magic_mix/1/vitra-vla-3b.pt" \
#     --eval_each_dataset \
#     --eval_sampler_step 161000 \
#     --eval_batches 200 \
#     --output_jsonl eval_results/pretrain_TB512_hf_85k_eval.jsonl \
#     --num_workers 32

# !python scripts/evaluate_pretrained_loss.py \
#     --config vitra/configs/human_pretrain.json \
#     --weights "data/vla_checkpoint/vitra_vla_3b/checkpoints/pretrain_TB512_B64_bf16True/checkpoints/final-epoch=0-step=20000.ckpt" \
#     --eval_each_dataset \
#     --eval_sampler_step 161000 \
#     --eval_batches 200 \
#     --output_jsonl eval_results/pretrain_TB512_20k_eval.jsonl \
#     --num_workers 32

# !zip -qr ../eval_results.zip eval_results/

## Test split evaluation (new protocol)

In [ ]:
# Dry run to determinate the suitable cutoff step
# !python scripts/evaluate_pretrained_loss.py \
#   --config vitra/configs/human_pretrain.json \
#   --dry_run \
#   --sweep_cutoff_steps 96000,112000,128000,144000,160000 \
#   --output_jsonl ../dry_run.jsonl

In [ ]:
!mkdir eval_test_split
CUTOFF = 128000
EVAL_BATCHES = 16400

!python scripts/evaluate_pretrained_loss.py \
    --config vitra/configs/human_pretrain.json \
    --weights "/kaggle/input/models/ldthanh/vitra-vla-3b/transformers/magic_mix/1/vitra-vla-3b.pt" \
    --eval_dataset epic \
    --eval_sampler_step {CUTOFF} \
    --eval_batches {EVAL_BATCHES} \
    --seen_sampler_steps {CUTOFF} \
    --output_jsonl eval_test_split/test_hf85k.jsonl

# !python scripts/evaluate_pretrained_loss.py \
#     --config vitra/configs/human_pretrain.json \
#     --weights "/kaggle/input/models/ldthanh/vitra-vla-3b/transformers/magic_mix_epic_ssv2/8/final-epoch=0-step=16000.ckpt/weights.pt" \
#     --eval_dataset epic \
#     --eval_sampler_step {CUTOFF} \
#     --eval_batches {EVAL_BATCHES} \
#     --seen_sampler_steps {CUTOFF} \
#     --output_jsonl eval_test_split/test_step16k.jsonl

!zip -qr ../eval_results.zip eval_test_split/